# UPPP 135 - Week 8: Segregation

<a target="_blank" href="https://colab.research.google.com/github/knaaptime/uppp135-winter26-assn/blob/main/week8/segregation.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

In [ ]:
import pandas as pd
import geopandas as gpd
import segregation as seg
import matplotlib.pyplot as plt
from fsspec import filesystem

## Data

In [ ]:
fs = filesystem("https")
ca = gpd.read_parquet("https://github.com/oturns/example_datasets/raw/refs/heads/main/acs/ca_blockgroups_2021.pq", filesystem=fs)

In [ ]:
ca = ca[ca.n_total_pop>0]
ca['bw'] = ca.n_nonhisp_black_persons + ca.n_nonhisp_white_persons

In [ ]:
# count of total people has n_ prefix
race_cols = ['n_nonhisp_white_persons', 'n_nonhisp_black_persons', 'n_hispanic_persons', 'n_asian_persons']

# percentages have p_ prefix
pct_cols = ['p_nonhisp_white_persons', 'p_nonhisp_black_persons', 'p_hispanic_persons', 'p_asian_persons']

In [ ]:
ca[race_cols].sum() / ca['n_total_pop'].sum()

In [ ]:
(ca[race_cols].sum() / ca['n_total_pop'].sum()).sum()

In [ ]:
la_region = ca[ca.geoid.str[:5].isin(['06059', '06037'])]
la_region = la_region.to_crs(la_region.estimate_utm_crs())

the LA region is similar to the state overall, though less white

In [ ]:
la_region[race_cols].sum() / la_region['n_total_pop'].sum()

In [ ]:
(la_region[race_cols].sum() / la_region['n_total_pop'].sum()).sum()

In [ ]:
f, ax = plt.subplots(2,2, figsize=(8,10))
ax = ax.flatten()

for i, col in enumerate(pct_cols):
    la_region.plot(col, scheme='fisher_jenks', ax=ax[i])
    ax[i].set_title(col)
    ax[i].axis('off')
plt.tight_layout()

In [ ]:
from IPython.display import IFrame

IFrame("https://knaaptime.com/uppp135-winter26/lectures/week8/la_race_density.html", width=900, height=700)

In [ ]:
la = la_region[la_region.geoid.str.startswith('06037')]

oc = la_region[la_region.geoid.str.startswith('06059')]

In [ ]:
la.shape

But *within* the LA region, the two counties look a bit different

In [ ]:
la[race_cols].sum() / la['n_total_pop'].sum()

In [ ]:
oc[race_cols].sum() / oc['n_total_pop'].sum()

## Segregation Indices

In a famous article, Massey and Denton propose that among the dozens of segregation indices in the literature, they  ultimately capture 5 dimensions of segregation, which they label

- evenness
- isolation/exposure
- concentration
- clustering
- centralization

In following work, [Reardon et al](http://link.springer.com/10.1353/dem.0.0019) argue the concentration and clustering are a function of *spatial scale*, rather than dimensions in their own right. [Reardon and Firebaugh](http://journals.sagepub.com/doi/10.1111/1467-9531.00110) also argue we should focus on the evenness and isolation indices, specfically using Entropy/Information theory-based indices and Simpson's Interaction index. Centralization is a distinct concept focusing on *where* in a region minority populations locate, so we leave that aside for now.


> We conclude that the spatial exposure/isolation index $\tilde{P}\ast$-—which can be interpreted as a measure of the average composition of individuals’ local spatial environments—and the spatial information theory index $\tilde{H}$—-which can be interpreted as a measure of the variation in the diversity of the local spatial environments of each individual—are the most conceptually and mathematically satisfactory of the proposed spatial indices. -- [Reardon and O'Sullivan (2004)](http://doi.wiley.com/10.1111/j.0081-1750.2004.00150.x) [@reardon2004MeasuresSpatial]

from [Massey and Denton (1988)](http://www.jstor.org/stable/2579183)

- ***evenness***: Evenness refers to the differential distribution of two social groups am areal units in a city. A minority group is said to be segregated if it is unevenly distributed over areal units (Blau 1977). Evenness is not measured in any absolute sense, but is scaled relative to some other group. Evenness is maximized and segregation minimized when all units have the same relative number of minority and majority members as the city as a whole. Conversely, evenness is minimized, and segregation maximized, when no minority and majority members share a common area of residence.

- ***exposure***: Residential exposure refers to the degree of potential contact, or the possibility of interaction, between minority and majority group members within geographic areas of a city. Indices of exposure measure the extent to which minority and majority members physically confront one another by virtue of sharing a common residential area. For any city, the degree of minority exposure to the majority may be conceptualized as the likelihood of their sharing the same neighborhood. Rather than measuring segregation as departure from some abstract ideal of "evenness," exposure indices attempt to measure the experience of segregation as felt by the average minority or majority member

**TL DR**:  if population is *even*, then the share of people in each group in all neighborhoods should be similar to the population shares at the region level.

i.e. if the LA region is 28% white, then each neighborhood should be roughly 28% white

## Single Group Measures

In [ ]:
names = ['region', 'la', 'oc']

In [ ]:
la_region_bw = seg.singlegroup.Entropy(la_region,group_pop_var='n_nonhisp_black_persons', total_pop_var='bw' )
la_bw = seg.singlegroup.Entropy(la ,group_pop_var='n_nonhisp_black_persons', total_pop_var='bw' )
oc_bw =  seg.singlegroup.Entropy(oc ,group_pop_var='n_nonhisp_black_persons', total_pop_var='bw' )



### Black/white

In [ ]:
la_region_bw.statistic

In [ ]:
la_bw.statistic

In [ ]:
oc_bw.statistic

### Hispanic/non-Hispanic

In [ ]:
la_region_hnh = seg.singlegroup.Entropy(la_region,group_pop_var='n_hispanic_persons', total_pop_var='n_total_pop' )
la_hnh = seg.singlegroup.Entropy(la ,group_pop_var='n_hispanic_persons', total_pop_var='n_total_pop' )
oc_hnh =  seg.singlegroup.Entropy(oc ,group_pop_var='n_hispanic_persons', total_pop_var='n_total_pop' )

In [ ]:
la_region_hnh.statistic

In [ ]:
la_hnh.statistic

In [ ]:
oc_hnh.statistic

The single group `Interaction` index captures the *exposure* dimension. Lets compute H/nH segregation using that measure. The full path to the class is `seg.singlegroup.Interaction`.

Which county has a larger segregation measure?

### Spatial Measures

In [ ]:
la_region_hnh_spatial = seg.singlegroup.Entropy(la_region,group_pop_var='n_hispanic_persons', total_pop_var='n_total_pop', distance=2000)
la_hnh_spatial = seg.singlegroup.Entropy(la ,group_pop_var='n_hispanic_persons', total_pop_var='n_total_pop' , distance=2000)
oc_hnh_spatial =  seg.singlegroup.Entropy(oc ,group_pop_var='n_hispanic_persons', total_pop_var='n_total_pop' , distance=2000)

In [ ]:
la_region_hnh_spatial.statistic

In [ ]:
la_hnh_spatial.statistic

In [ ]:
oc_hnh_spatial.statistic

In [ ]:
pd.Series(
    dict(
        zip(
            names,
            [
                la_region_hnh_spatial.statistic,
                la_hnh_spatial.statistic,
                oc_hnh_spatial.statistic,
            ],
        )
    )
)

## Multi Group Measures

In [ ]:
la_region_multigroup = seg.multigroup.MultiInfoTheory(data=la_region, groups=race_cols)
la_multigroup = seg.multigroup.MultiInfoTheory(data=la, groups=race_cols)
oc_multigroup = seg.multigroup.MultiInfoTheory(data=oc, groups=race_cols)

In [ ]:
pd.Series(
    dict(
        zip(
            names,
            [
                la_region_multigroup.statistic,
                la_multigroup.statistic,
                oc_multigroup.statistic,
            ],
        )
    )
)

## Spatial Scale

In [ ]:
dists = list(range(500,4500, 500))

In [ ]:
dists

In [ ]:
oc.crs

In [ ]:
from libpysal.graph import Graph

f, ax = plt.subplots(2, 2, figsize=(8, 8))
ax = ax.flatten()
for i, d in enumerate([500, 1000, 1500, 2000]):
    oc.plot(ax=ax[i], alpha=0.3, linewidth=0.5)
    Graph.build_distance_band(oc.set_geometry(oc.centroid), threshold=d).plot(
        oc,
        ax=ax[i],
        nodes=False,
        edge_kws={
            "linewidth": 0.4,
        },
    )
    ax[i].set_title(f"Distance: {d}m")
    ax[i].axis("off")
plt.tight_layout()

In [ ]:
profile_multigroup_la = seg.dynamics.compute_multiscalar_profile(
    gdf=la,
    segregation_index=seg.multigroup.MultiInfoTheory,
    groups=race_cols,
    distances=dists,
)

In [ ]:

profile_multigroup_oc = seg.dynamics.compute_multiscalar_profile(
    gdf=oc, 
    segregation_index=seg.multigroup.MultiInfoTheory, 
    groups=race_cols, 
    distances=dists,
)

In [ ]:
profile_multigroup_la

In [ ]:
profile_multigroup_la.rename('LA').plot(legend=True)
profile_multigroup_oc.rename('OC').plot(legend=True)

## Inference

### Single Group

We already have several indices computed and ready for us to use. Lets use the Hispanic/non-Hispanic entropy-based indices and ask whether one county is statistically more segrgated than the other.

In [ ]:
la_hnh.statistic

In [ ]:
oc_hnh.statistic

In [ ]:
la_hnh.statistic - oc_hnh.statistic

0.013 is a small difference! Is that small enough to happen by chance?

We have two segregation indices, one each for LA and OC, so this is a *two-value* test. We want to ask whether segregation is larger in one county than the other, and we can test this using a forma hypothesis.

If $Seg_{la}$ is the segregation index for LA county and $Seg_{oc}$ is the segregation index for Orange County, then

- $H_0: Seg_{la} = Seg_{oc}$

- $H_A: Seg_{la} \neq Seg_{oc}$

In [ ]:
county_hnh_comparison = seg.inference.TwoValueTest(la_hnh, oc_hnh, iterations_under_null=9999)

In [ ]:
county_hnh_comparison.p_value

In [ ]:
county_hnh_comparison.plot()

### Multigroup

In [ ]:
la_multigroup.statistic

In [ ]:
oc_multigroup.statistic

In [ ]:
la_multigroup.statistic - oc_multigroup.statistic

is 0.06 a "big" difference? Big enough that it's not due to chance?

- $H_0: Seg_{la} = Seg_{oc}$

- $H_A: Seg_{la} \neq Seg_{oc}$

In [ ]:
county_multigroup_comparison = seg.inference.TwoValueTest(la_multigroup, oc_multigroup, iterations_under_null=9999, n_jobs=-1)

In [ ]:
county_multigroup_comparison.p_value

In [ ]:
county_multigroup_comparison.plot()